# Transformer Model - Hyper Parameter Tuning

In [ ]:
import numpy as np
import pandas as pd
import torch
from sklearn.preprocessing import MinMaxScaler, StandardScaler
import os
import sys
sys.path.append("..")


from utils import load_and_prepare_data, get_error_df
root = os.path.join(os.path.dirname(os.path.abspath(__file__)), '..', 'Data')
denoised_sales_df = pd.read_excel(os.path.join(root, 'FixedData_Random_Smoothed.xlsx'))
noised_sales_df = pd.read_excel(os.path.join(root, 'DataMissingValuesFilled.xlsx'))
holiday_df = pd.read_csv(os.path.join(root, 'holidays.csv'))
crude_df = pd.read_csv(os.path.join(root, 'Crude Oil Prices.csv'))


val_split_date = '2023-06-01'
test_split_date = '2024-01-01'
seq_length = 30
forecast_length = 30

epochs = 150

shift_crude_days = 0
warmup_days = 14
warmup_epochs = 2
warmup_every_n_days = 1

loss_fn = 'asymmetric_mse'

In [ ]:
from itertools import product

N_RUNS = 20

EXPERIMENTS = [
    # -------------------- SALES ONLY --------------------
    {
        "name": "sales_only_scaled_no_calender",
        "use_scaling": True,
        "use_crude": False,
        "use_month": False,
        "use_month_sin_cos": False,
        "use_dayofweek": False,
        "use_dayofweek_sin_cos": False,
    },
    {
        "name": "sales_only_scaled_calender_numbers",
        "use_scaling": True,
        "use_crude": False,
        "use_month": True,
        "use_month_sin_cos": False,
        "use_dayofweek": True,
        "use_dayofweek_sin_cos": False,
    },
    {
        "name": "sales_only_scaled_calender_sincos",
        "use_scaling": True,
        "use_crude": False,
        "use_month": False,
        "use_month_sin_cos": True,
        "use_dayofweek": False,
        "use_dayofweek_sin_cos": True,
    },

    # -------------------- SALES + CRUDE --------------------

    {
        "name": "sales_and_crude_scaled_no_calender",
        "use_scaling": True,
        "use_crude": True,
        "use_month": False,
        "use_month_sin_cos": False,
        "use_dayofweek": False,
        "use_dayofweek_sin_cos": False,
    },
    {
        "name": "sales_and_crude_scaled_calender_numbers",
        "use_scaling": True,
        "use_crude": True,
        "use_month": True,
        "use_month_sin_cos": False,
        "use_dayofweek": True,
        "use_dayofweek_sin_cos": False,
    },
    {
        "name": "sales_and_crude_scaled_calender_sincos",
        "use_scaling": True,
        "use_crude": True,
        "use_month": False,
        "use_month_sin_cos": True,
        "use_dayofweek": False,
        "use_dayofweek_sin_cos": True,
    },
]


In [3]:
def build_datasets_for_config(cfg):
  sales_scaler = MinMaxScaler() if cfg["use_scaling"] else None
  crude_scaler = MinMaxScaler() if cfg["use_scaling"] else None

  train_dataset, val_dataset, test_dataset = load_and_prepare_data(
      denoised_sales_df=denoised_sales_df,
      noised_sales_df=noised_sales_df,
      crude_df=crude_df,
      val_split_date=val_split_date,
      test_split_date=test_split_date,
      seq_length=seq_length,
      forecast_length=forecast_length,
      shift_crude_days=shift_crude_days,
      use_crude=cfg["use_crude"],
      use_month=cfg["use_month"],
      use_month_sin_cos=cfg["use_month_sin_cos"],
      use_dayofweek=cfg["use_dayofweek"],
      use_dayofweek_sin_cos=cfg["use_dayofweek_sin_cos"],
      sales_scaler=sales_scaler,
      crude_scaler=crude_scaler
  )

  return train_dataset, val_dataset, test_dataset

def compute_mape(y_true, y_pred, eps=1e-8):
    Y_true = np.array(y_true.copy(), dtype=float)
    Y_pred = np.array(y_pred.copy(), dtype=float)

    # Avoid division by zero
    epsilon=1e-8
    Y_true = np.where(Y_true == 0, epsilon, Y_true)

    # Absolute percentage error
    ape = np.abs((Y_true - Y_pred)*100 / Y_true)

    # Mean over horizon (axis=1)
    mapes = np.mean(ape, axis=1)

    return mapes.mean()


In [4]:
!pip install optuna

In [ ]:
from engine import train_model, generate_predictions
from holidayCorrection import holiday_correction
from Models import Transformer
import optuna

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
results = {}
for cfg in EXPERIMENTS:
    train_dataset, val_dataset, test_dataset = build_datasets_for_config(cfg)
    input_dim = train_dataset.X.shape[2]
    print(f"Starting study for {cfg['name']}")
    def objective(trial):
    
        # ----------------- Hyperparameters -----------------
        num_layers = trial.suggest_int("num_layers", 1, 8)
        d_model = trial.suggest_categorical("d_model", [32, 64, 128, 256,512])
        possible_heads = [h for h in [1, 2, 4, 8] if d_model % h == 0]
        n_head = trial.suggest_categorical("n_head", possible_heads)
        dim_feedforward = 4*d_model
        dropout = trial.suggest_float('dropout', 0.0, 0.5)
        lr = trial.suggest_float('lr', 1e-4, 5e-3, log=True)
        batch_size = trial.suggest_categorical('batch_size', [16, 32, 64])
        under_parameter = trial.suggest_float('under_parameter', 0, 2)
        over_parameter = trial.suggest_float('over_parameter', 0, 2)
        output_length = forecast_length
        activation = 'relu'
    
    
        # ----------------- Dataloaders -----------------
        train_loader = train_dataset.get_dataloader(batch_size=batch_size, shuffle=True)
        val_loader = val_dataset.get_dataloader(batch_size=batch_size, shuffle=False)
    
        # ----------------- Build Transformer -----------------
        model = Transformer(
            input_dim=input_dim,
            d_model=d_model,
            nhead=n_head,
            num_layers=num_layers,
            dim_feedforward=dim_feedforward,
            output_length=output_length,
            dropout=dropout
        )

        # ----------------- Train -----------------
        _, val_losses, _, _ = train_model(
            model=model,
            train_loader=train_loader,
            val_loader=val_loader,
            under_parameter=under_parameter,
            over_parameter=over_parameter,
            epochs=100,
            lr=lr,
            device=device,
            horizon_start=0,
            horizon_end=train_dataset.Y.shape[1],
            patience=20,
            loss_fn=loss_fn
        )
    
        # ----------------- Generate predictions -----------------
        Y_pred_uncorrected, Y_true, Y_true_noised = generate_predictions(model, test_dataset, device=device)
        Y_pred_corrected = holiday_correction(Y_pred_uncorrected, test_dataset.sample_dates, noised_sales_df, holiday_df)
    
        return compute_mape(y_true=Y_true_noised, y_pred=Y_pred_corrected)

    study = optuna.create_study(direction='minimize')
    study.optimize(objective, n_trials=75)  # run 20 trials
    params = study.best_trial.params

    results[cfg["name"]] = {
        "bestMape" : study.best_value,
        "num_layers" : params.get('num_layers', None),
        "d_model" : params.get('d_model',None),
        "n_head" : params.get('n_head',None),
        "dropout" : params.get('dropout', 0.0),
        "lr" : params.get('lr', None),
        "batch_size" : params.get('batch_size', None),
        "under_parameter" : params.get('under_parameter', None),
        "over_parameter" : params.get('over_parameter', None),
    }


/kaggle/input/datasets/milanroy/cdis-codebase-and-dataset/CodeBase and Dataset/CodeBase/utils.py:22: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['Date'] = pd.to_datetime(df['Date'])
[I 2026-02-15 20:07:07,212] A new study created in memory with name: no-name-d2e4b991-dab4-4a06-8f99-4ef5dc12d481


Total records : 1736
Train records : 1156
Val records   : 214
Test records  : 366

Val starts  at: 2023-06-01
Test starts at: 2024-01-01
Using features: ['Sales']
Starting study for sales_only_no_scaled_no_calender
Epoch 10/100 | Train Loss: 2337718928.695652 | Val Loss: 1217520976.000000
Epoch 20/100 | Train Loss: 2054279511.188406 | Val Loss: 767544508.000000
Epoch 30/100 | Train Loss: 1979772876.985507 | Val Loss: 1005136232.000000

Early stopping triggered at epoch 37. Best Val Loss: 389558321.333333


[I 2026-02-15 20:07:25,893] Trial 0 finished with value: 10.180854430707113 and parameters: {'num_layers': 5, 'layer_size_1': 1280, 'layer_size_2': 1280, 'layer_size_3': 1792, 'layer_size_4': 1344, 'layer_size_5': 256, 'dropout': 0.34188027951737354, 'lr': 0.00017730326881222204, 'batch_size': 16, 'under_parameter': 1.2004080104553598, 'over_parameter': 0.47594180322925417}. Best is trial 0 with value: 10.180854430707113.


Epoch 10/100 | Train Loss: 1127294186.057143 | Val Loss: 816554133.333333
Epoch 20/100 | Train Loss: 957296830.171429 | Val Loss: 715910149.333333
Epoch 30/100 | Train Loss: 924368555.885714 | Val Loss: 338534857.333333
Epoch 40/100 | Train Loss: 843904486.400000 | Val Loss: 555230709.333333
Epoch 50/100 | Train Loss: 886849413.485714 | Val Loss: 337520261.333333
Epoch 60/100 | Train Loss: 805801186.742857 | Val Loss: 344664289.333333
Epoch 70/100 | Train Loss: 835456248.685714 | Val Loss: 493463586.666667

Early stopping triggered at epoch 70. Best Val Loss: 337520261.333333


[I 2026-02-15 20:07:32,385] Trial 1 finished with value: 7.433169494610295 and parameters: {'num_layers': 2, 'layer_size_1': 960, 'layer_size_2': 1408, 'dropout': 0.22662888438785256, 'lr': 0.0001750981480049268, 'batch_size': 32, 'under_parameter': 0.5401478827406161, 'over_parameter': 1.6545778773487991}. Best is trial 1 with value: 7.433169494610295.


...

[I 2026-02-15 23:13:22,927] Trial 147 finished with value: 10.965829827676 and parameters: {'num_layers': 1, 'layer_size_1': 320, 'dropout': 0.4776456829836569, 'lr': 0.0016708974898207425, 'batch_size': 32, 'under_parameter': 1.5945842338901368, 'over_parameter': 0.3652252377879856}. Best is trial 19 with value: 7.324546181821247.


Epoch 10/100 | Train Loss: 0.003806 | Val Loss: 0.002898
Epoch 20/100 | Train Loss: 0.002666 | Val Loss: 0.001788

Early stopping triggered at epoch 26. Best Val Loss: 0.001236


[I 2026-02-15 23:13:26,789] Trial 148 finished with value: 8.392329113026678 and parameters: {'num_layers': 1, 'layer_size_1': 448, 'dropout': 0.4608869101050451, 'lr': 0.002066772388604989, 'batch_size': 16, 'under_parameter': 0.33247674830579627, 'over_parameter': 0.4218010676998023}. Best is trial 19 with value: 7.324546181821247.


Epoch 10/100 | Train Loss: 0.008707 | Val Loss: 0.005295
Epoch 20/100 | Train Loss: 0.006301 | Val Loss: 0.005540

Early stopping triggered at epoch 24. Best Val Loss: 0.002780


[I 2026-02-15 23:13:30,430] Trial 149 finished with value: 8.683908279841322 and parameters: {'num_layers': 1, 'layer_size_1': 512, 'dropout': 0.438382825240741, 'lr': 0.002273944752587242, 'batch_size': 16, 'under_parameter': 0.5716814574789866, 'over_parameter': 1.8720872688274952}. Best is trial 19 with value: 7.324546181821247.


In [ ]:
import json

with open('Transformer Hyperparameters.txt', 'w') as f:
    json.dump(results, f, indent=4)
